# MBWR Fugacity Coefficient Calculator
### Pure CO₂ · Pure N₂ · Binary CO₂+N₂ mixture · LAMMPS output

**Equation of state:** Modified Benedict–Webb–Rubin (MBWR), Johnson, Zollweg & Gubbins (1993)  
**Units:** P [atm], ρ [mol/L], T [K]

$$P^* = \rho^* T^* + \sum_{i=1}^{8} a_i \rho^{*(i+1)} + F \sum_{i=1}^{6} b_i \rho^{*(2i+1)}, \quad F = e^{-\gamma \rho^{*2}}$$

**Source:**  
- J. K. Johnson, J. A. Zollweg, K. E. Gubbins, *Mol. Phys.* **78**(3), 591–618 (1993)  
- LJ parameters derived from NIST critical properties + JZG critical point (T\*c=1.313, ρ\*c=0.310)  
- vdW1 mixing rules: Eqs. 20–21 of JZG; partial φ via numerical ∂A^res/∂N_k at const. T, V

---

---
## Cell 1: MBWR core functions (run once)

In [ ]:
import math
from scipy.optimize import brentq

BAR_TO_ATM = 0.986923
kB  = 1.380649e-23    # J/K
NA  = 6.02214076e23   # 1/mol

# ── JZG (1993) Table 10: 32 MBWR parameters (1-indexed) ─────────────────
X = [None,                     # x[0] unused (1-indexing)
      0.8623085097507421,       # x1
      2.976218765822098,        # x2
     -8.402230115796038,        # x3
      0.1054136629203555,       # x4
     -0.8564583828174598,       # x5
      1.582759470107601,        # x6
      0.7639421948305453,       # x7
      1.753173414312048,        # x8
      2798.291772190376,        # x9
     -0.04839422026085765,      # x10
      0.9963265197721935,       # x11
    -36.98000291272493,         # x12
     20.84012299434647,         # x13
     83.05402124717285,         # x14
   -957.4799715203068,          # x15
   -147.7746229234994,          # x16
     63.98607852471505,         # x17
     16.03993673294834,         # x18
     68.05916615864377,         # x19
  -2791.293578795945,           # x20
     -6.245128304568454,        # x21
  -8116.836104958410,           # x22
     14.88735559561229,         # x23
 -10593.46754655084,            # x24
   -113.1607632802822,          # x25
  -8867.771540418822,           # x26
    -39.86982844450543,         # x27
  -4689.270299917261,           # x28
    259.3535277438717,          # x29
  -2694.523589434903,           # x30
   -721.8487631550215,          # x31
    172.1802063863269,          # x32
]
GAMMA = 3.0   # nonlinear parameter (JZG Sect. 4.1, fixed for historical continuity)

# ── LJ parameters from critical properties (JZG T*c=1.313, rho*c=0.310) ─
EPS_K = {}   # eps/k_B [K]
SIG   = {}   # sigma   [m]

def lj_from_critical(Tc_K, rhoc_kgm3, M_gmol):
    """Single-site LJ eps/k [K] and sigma [m] from critical properties."""
    eps_k = Tc_K / 1.313
    rhoc_mol_m3 = rhoc_kgm3 / (M_gmol * 1e-3)
    sig3  = 0.310 / (NA * rhoc_mol_m3)
    return eps_k, sig3 ** (1.0 / 3.0)

EPS_K['CO2'], SIG['CO2'] = lj_from_critical(304.13, 467.6,  44.01)
EPS_K['N2'],  SIG['N2']  = lj_from_critical(126.19, 313.3,  28.014)

# ── Temperature-dependent coefficients (Tables 5 & 6) ────────────────────
def _ai(Ts):
    return [
        X[1]*Ts + X[2]*Ts**0.5 + X[3] + X[4]/Ts + X[5]/Ts**2,   # a1
        X[6]*Ts + X[7] + X[8]/Ts + X[9]/Ts**2,                    # a2
        X[10]*Ts + X[11] + X[12]/Ts,                               # a3
        X[13],                                                       # a4
        X[14]/Ts + X[15]/Ts**2,                                    # a5
        X[16]/Ts,                                                   # a6
        X[17]/Ts + X[18]/Ts**2,                                    # a7
        X[19]/Ts**2,                                               # a8
    ]

def _bi(Ts):
    return [
        X[20]/Ts**2 + X[21]/Ts**3,                          # b1
        X[22]/Ts**2 + X[23]/Ts**4,                          # b2
        X[24]/Ts**2 + X[25]/Ts**3,                          # b3
        X[26]/Ts**2 + X[27]/Ts**4,                          # b4
        X[28]/Ts**2 + X[29]/Ts**3,                          # b5
        X[30]/Ts**2 + X[31]/Ts**3 + X[32]/Ts**4,           # b6
    ]

def _Gi_F(rhos):
    """Density-dependent G_i functions (Table 7) and F = exp(-gamma*rho*^2)."""
    g = GAMMA; F = math.exp(-g * rhos**2)
    G = [0.0] * 7
    G[1] = (1 - F) / (2*g)
    G[2] = -(F*rhos**2  - 2*G[1])  / (2*g)
    G[3] = -(F*rhos**4  - 4*G[2])  / (2*g)
    G[4] = -(F*rhos**6  - 6*G[3])  / (2*g)
    G[5] = -(F*rhos**8  - 8*G[4])  / (2*g)
    G[6] = -(F*rhos**10 - 10*G[5]) / (2*g)
    return G, F

# ── Eq. 7: reduced pressure P*(rho*, T*) ────────────────────────────────
def Pstar(rhos, Ts):
    a = _ai(Ts); b = _bi(Ts); G, F = _Gi_F(rhos)
    P = rhos * Ts
    for i, ai in enumerate(a, 1): P += ai * rhos**(i + 1)
    for i, bi in enumerate(b, 1): P += F * bi * rhos**(2*i + 1)
    return P

# ── Eq. 5: residual Helmholtz A_r*(rho*, T*) ────────────────────────────
def Astar(rhos, Ts):
    a = _ai(Ts); b = _bi(Ts); G, _ = _Gi_F(rhos)
    return (sum(a[i-1] * rhos**i / i for i in range(1, 9)) +
            sum(b[i-1] * G[i]        for i in range(1, 7)))

# ── vdW1 mixing rules (JZG Eqs. 20–21) ───────────────────────────────────
def vdw1_mix(x_dict, eps_cross, sig_cross):
    """
    Returns (sig3_x [m³], eps_x [K]).
    eps_cross / sig_cross: dicts keyed by (species_i, species_j).
    """
    names = list(x_dict)
    def _ec(a, b): return eps_cross.get((a,b), eps_cross.get((b,a), None))
    def _sc(a, b): return sig_cross.get((a,b), sig_cross.get((b,a), None))
    sig3x = sum(x_dict[i]*x_dict[j]*_sc(i,j)**3
                for i in names for j in names)
    epsx  = sum(x_dict[i]*x_dict[j]*_ec(i,j)*_sc(i,j)**3
                for i in names for j in names) / sig3x
    return sig3x, epsx

# ── Root-finding: rho* from P*(rho*, T*) = P*_target ─────────────────────
def _solve_rhostar(Ts, Pst):
    return brentq(lambda rs: Pstar(rs, Ts) - Pst, 1e-9, 0.85, xtol=1e-12)

# ── Pure-component fugacity coefficient (Helmholtz route, Eq. 10) ─────────
def mbwr_pure(T_K, P_bar, species):
    """
    Returns (phi, rho_mol_L, Z, f_atm) for a pure LJ gas.
    ln(phi) = A_r*/T* + Z - 1 - ln(Z)   [from Eq. 5 + Eq. 10 of JZG]
    """
    P_Pa  = P_bar * 1e5
    ek    = EPS_K[species]; sig = SIG[species]; sig3 = sig**3
    Ts    = T_K / ek
    Ps    = P_Pa * sig3 / (kB * ek)
    rhos  = _solve_rhostar(Ts, Ps)
    Z     = Ps / (rhos * Ts)
    Ar    = Astar(rhos, Ts)
    phi   = math.exp(Ar/Ts + Z - 1.0 - math.log(Z))
    rho_mol_L = rhos / (NA * sig3 * 1e3)
    f_atm = phi * P_bar * BAR_TO_ATM
    return phi, rho_mol_L, Z, f_atm

# ── Mixture partial fugacity (numerical dA^res/dN_k at const. T, V) ───────
def mbwr_mixture(T_K, P_bar, x_dict, eps_cross, sig_cross):
    """
    Returns (mix_results, rho_mol_L, Z).
    mix_results = {species: (phi_k, f_k_atm)}.
    Partial fugacity from: ln(phi_k) = mu_k^res/(kT) - ln(Z)
    where mu_k^res = dA^res/dN_k at const. T, V  (numerical derivative).
    """
    P_Pa           = P_bar * 1e5
    sig3x, epsx    = vdw1_mix(x_dict, eps_cross, sig_cross)
    Ts             = T_K / epsx
    Ps             = P_Pa * sig3x / (kB * epsx)
    rhos           = _solve_rhostar(Ts, Ps)
    Z              = Ps / (rhos * Ts)
    rho_mol_L      = rhos / (NA * sig3x * 1e3)
    rho_num        = rhos / sig3x                  # number density [1/m³]

    def total_Ares(N_dict):
        """Total residual Helmholtz [J] at constant T and V for N-particle system."""
        N_tot   = sum(N_dict.values())
        x_new   = {k: N_dict[k]/N_tot for k in N_dict}
        s3, ex  = vdw1_mix(x_new, eps_cross, sig_cross)
        rs_new  = (N_tot * rho_num) * s3            # const. V → rho_new = N_new/V
        Ts_new  = T_K / ex
        return N_tot * ex * kB * Astar(rs_new, Ts_new)

    N0   = {k: x_dict[k] for k in x_dict}          # pseudo-particle counts = x_k
    A0   = total_Ares(N0)
    dN   = 1e-5
    res  = {}
    for k in x_dict:
        N1 = {s: x_dict[s] for s in x_dict}
        N1[k] += dN
        mu_res_k = (total_Ares(N1) - A0) / dN      # J per pseudo-particle
        ln_phi_k = mu_res_k / (kB * T_K) - math.log(Z)
        phi_k    = math.exp(ln_phi_k)
        f_k_atm  = phi_k * x_dict[k] * P_bar * BAR_TO_ATM
        res[k]   = (phi_k, f_k_atm)

    return res, rho_mol_L, Z

print('\u2713 MBWR functions loaded  (JZG 1993, Table 10, gamma=3)')
print(f'  CO2: eps/k = {EPS_K["CO2"]:.2f} K,  sigma = {SIG["CO2"]*1e10:.4f} \u00c5')
print(f'  N2 : eps/k = {EPS_K["N2"]:.2f} K,  sigma = {SIG["N2"]*1e10:.4f} \u00c5')

---
## Cell 2: Settings  *(edit here)*

In [ ]:
T_K        = 343.15   # temperature [K]

P_bar_list = [0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5,
              0.6, 0.7, 0.8, 0.9, 1.0,
              2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 15.0, 20.0]   # [bar]

# Binary mixture composition
y_CO2    = 0.15   # mole fraction CO2
y_N2     = 0.85   # mole fraction N2
k_CO2_N2 = 0.02   # binary interaction parameter  eps_12 = (1-k12)*sqrt(eps1*eps2)

# Cross-interaction parameters (Lorentz-Berthelot + kij on energy)
eps_cross = {('CO2', 'N2'): (1 - k_CO2_N2) * math.sqrt(EPS_K['CO2'] * EPS_K['N2'])}
sig_cross = {('CO2', 'N2'): (SIG['CO2'] + SIG['N2']) / 2.0}
# Also need self-pairs for vdw1_mix
for sp in ('CO2', 'N2'):
    eps_cross[(sp, sp)] = EPS_K[sp]
    sig_cross[(sp, sp)] = SIG[sp]

y_dict = {'CO2': y_CO2, 'N2': y_N2}

print(f'T = {T_K} K,  {len(P_bar_list)} pressure points,  '
      f'mixture: {y_CO2:.0%} CO2 + {y_N2:.0%} N2,  k12 = {k_CO2_N2}')

---
## Cell 3: Pure CO₂

In [ ]:
res_co2 = []

print(f"{'P[bar]':>7}  {'P[atm]':>8}  {'rho[mol/L]':>11}  {'Z':>8}  {'phi':>10}  {'f[atm]':>9}")
print('-'*65)

for Pb in P_bar_list:
    Pa          = Pb * BAR_TO_ATM
    phi, rho, Z, f = mbwr_pure(T_K, Pb, 'CO2')
    print(f"{Pb:>7.2f}  {Pa:>8.5f}  {rho:>11.7f}  {Z:>8.6f}  {phi:>10.7f}  {f:>9.6f}")
    res_co2.append(dict(P_bar=Pb, P_atm=Pa, rho=rho, Z=Z, phi=phi, f_atm=f))

---
## Cell 4: Pure N₂

In [ ]:
res_n2 = []

print(f"{'P[bar]':>7}  {'P[atm]':>8}  {'rho[mol/L]':>11}  {'Z':>8}  {'phi':>10}  {'f[atm]':>9}")
print('-'*65)

for Pb in P_bar_list:
    Pa          = Pb * BAR_TO_ATM
    phi, rho, Z, f = mbwr_pure(T_K, Pb, 'N2')
    print(f"{Pb:>7.2f}  {Pa:>8.5f}  {rho:>11.7f}  {Z:>8.6f}  {phi:>10.7f}  {f:>9.6f}")
    res_n2.append(dict(P_bar=Pb, P_atm=Pa, rho=rho, Z=Z, phi=phi, f_atm=f))

---
## Cell 5: Binary mixture CO₂ + N₂

In [ ]:
res_mix = []

print(f"{'P[bar]':>7}  {'rho[mol/L]':>11}  {'Z':>7}  "
      f"{'phi_CO2':>9}  {'f_CO2[atm]':>11}  {'phi_N2':>9}  {'f_N2[atm]':>10}")
print('-'*80)

for Pb in P_bar_list:
    Pa                  = Pb * BAR_TO_ATM
    mix_res, rho, Z     = mbwr_mixture(T_K, Pb, y_dict, eps_cross, sig_cross)
    phi1, f1            = mix_res['CO2']
    phi2, f2            = mix_res['N2']
    print(f"{Pb:>7.2f}  {rho:>11.7f}  {Z:>7.5f}  "
          f"{phi1:>9.7f}  {f1:>11.6f}  {phi2:>9.7f}  {f2:>10.6f}")
    res_mix.append(dict(P_bar=Pb, P_atm=Pa, rho=rho, Z=Z,
                        phi_CO2=phi1, f_CO2=f1,
                        phi_N2=phi2,  f_N2=f2))

---
## Cell 6: f [atm] lists  *(ready to copy into LAMMPS)*

In LAMMPS `fix gcmc` you pass:
- **Pure gas** → `pressure ${P_atm}  fugacity_coeff ${phi}`  
- **Mixture gas** → `pressure ${P_partial}  fugacity_coeff ${phi_k}`  
  where `P_partial = y_k * P_total` (already done below)

The fugacity `f = phi * P` (or `phi * y_k * P` for a mixture) is what enters the chemical potential **μ = RT ln(f)** inside GCMC.

In [ ]:
# ── Pure CO2 ────────────────────────────────────────────────────────────
print('=== Pure CO2, f [atm] list ===')
for r in res_co2:
    print(f"  P = {r['P_bar']:5.2f} bar  \u2192  phi = {r['phi']:.7f}   f = {r['f_atm']:.6f} atm")

print()

# ── Pure N2 ─────────────────────────────────────────────────────────────
print('=== Pure N2, f [atm] list ===')
for r in res_n2:
    print(f"  P = {r['P_bar']:5.2f} bar  \u2192  phi = {r['phi']:.7f}   f = {r['f_atm']:.6f} atm")

print()

# ── Binary mixture ───────────────────────────────────────────────────────
print('=== CO2+N2 mixture, partial f [atm] list ===')
print('    (use f_CO2 and f_N2 as the effective pressure in separate fix gcmc calls)')
for r in res_mix:
    print(f"  P = {r['P_bar']:5.2f} bar  \u2192  "
          f"phi_CO2 = {r['phi_CO2']:.7f}  f_CO2 = {r['f_CO2']:.6f} atm  |  "
          f"phi_N2  = {r['phi_N2']:.7f}  f_N2  = {r['f_N2']:.6f} atm")

---
## Cell 7: LAMMPS-ready variable blocks

Copy the printed block directly into your LAMMPS input script.  
Each pressure point is a `variable` you reference inside your pressure loop.

In [ ]:
def lammps_block(label, results, phi_key, f_key):
    """
    Print a LAMMPS variable block for a given species.
    Usage in LAMMPS loop:
        fix gcmc ... pressure ${P_atm} fugacity_coeff ${phi}
    """
    lines = [f'# \u2500\u2500 MBWR fugacity, {label}  T={T_K} K \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500']
    for i, r in enumerate(results, 1):
        lines.append(f'# point {i:02d}: P={r["P_bar"]:.2f} bar  '
                     f'phi={r[phi_key]:.7f}  f={r[f_key]:.6f} atm')
    lines.append('')

    for i, r in enumerate(results, 1):
        lines.append(f'variable phi_{label}_p{i:02d}  equal {r[phi_key]:.10f}   '
                     f'# P={r["P_bar"]:.2f} bar')
    lines.append('')

    for i, r in enumerate(results, 1):
        lines.append(f'variable Patm_{label}_p{i:02d} equal {r["P_atm"]:.6f}    '
                     f'# {r["P_bar"]:.2f} bar')

    print('\n'.join(lines))
    print()

lammps_block('CO2',    res_co2,  'phi',     'f_atm')
lammps_block('N2',     res_n2,   'phi',     'f_atm')
lammps_block('CO2mix', res_mix,  'phi_CO2', 'f_CO2')
lammps_block('N2mix',  res_mix,  'phi_N2',  'f_N2')

---
## Cell 8: Write `mbwr_phi_table.lmp`  *(include this file in LAMMPS)*

This writes a single file you can `include mbwr_phi_table.lmp` at the top of your LAMMPS script.  
Then in your pressure loop use e.g.:
```lammps
fix gcmc_CO2 CO2 gcmc ... pressure ${Patm_CO2_p01} fugacity_coeff ${phi_CO2_p01}
```

In [ ]:
outfile = 'mbwr_phi_table.lmp'

def write_block(fh, label, results, phi_key, f_key):
    fh.write(f'# \u2500\u2500 {label}  T={T_K} K \u2500\u2500\n')
    for i, r in enumerate(results, 1):
        fh.write(f'variable phi_{label}_p{i:02d}  equal {r[phi_key]:.10f}'
                 f'   # P={r["P_bar"]:.2f} bar  f={r[f_key]:.6f} atm\n')
    fh.write('\n')
    for i, r in enumerate(results, 1):
        fh.write(f'variable Patm_{label}_p{i:02d} equal {r["P_atm"]:.6f}'
                 f'    # {r["P_bar"]:.2f} bar\n')
    fh.write('\n')

with open(outfile, 'w') as fh:
    fh.write('# ================================================================\n')
    fh.write(f'# MBWR fugacity table (JZG 1993), auto-generated  T = {T_K} K\n')
    fh.write(f'# Mixture: {y_CO2:.0%} CO2 + {y_N2:.0%} N2   k12 = {k_CO2_N2}\n')
    fh.write('# Usage:  include mbwr_phi_table.lmp\n')
    fh.write('#   fix gcmc ... pressure ${Patm_CO2_p01} fugacity_coeff ${phi_CO2_p01}\n')
    fh.write('# ================================================================\n\n')
    write_block(fh, 'CO2',    res_co2,  'phi',     'f_atm')
    write_block(fh, 'N2',     res_n2,   'phi',     'f_atm')
    write_block(fh, 'CO2mix', res_mix,  'phi_CO2', 'f_CO2')
    write_block(fh, 'N2mix',  res_mix,  'phi_N2',  'f_N2')

print(f'\u2713 Written: {outfile}')

with open(outfile) as fh:
    print(fh.read())

---
## Cell 9: Pandas summary tables  *(optional)*

In [ ]:
import pandas as pd

df_co2 = pd.DataFrame(res_co2)[['P_bar','P_atm','rho','Z','phi','f_atm']]
df_n2  = pd.DataFrame(res_n2) [['P_bar','P_atm','rho','Z','phi','f_atm']]
df_mix = pd.DataFrame(res_mix)[['P_bar','P_atm','rho','Z',
                                  'phi_CO2','f_CO2','phi_N2','f_N2']]

print('=== Pure CO2 ==='); display(df_co2.style.format('{:.6f}'))
print('=== Pure N2  ==='); display(df_n2 .style.format('{:.6f}'))
print('=== Mixture  ==='); display(df_mix.style.format('{:.6f}'))

---
## Cell 10: Look up a single pressure point

In [ ]:
# Change this to whichever pressure you need
query_bar = 1.0

def lookup(results, P_bar, label):
    r = next((x for x in results if abs(x['P_bar']-P_bar)<1e-9), None)
    if r is None:
        print(f'{label}: P={P_bar} bar not found in list'); return
    print(f'{label} @ P={P_bar} bar:')
    for k,v in r.items():
        print(f'  {k:12s} = {v:.8f}')
    print()

lookup(res_co2, query_bar, 'Pure CO2')
lookup(res_n2,  query_bar, 'Pure N2')
lookup(res_mix, query_bar, 'CO2+N2 mixture')

In [ ]:
print("f_CO2 (pure)  :", " ".join(str(round(r['f_atm'],6)) for r in res_co2))
print("f_N2  (pure)  :", " ".join(str(round(r['f_atm'],6)) for r in res_n2))
print("f_CO2 (mix)   :", " ".join(str(round(r['f_CO2'],6)) for r in res_mix))
print("f_N2  (mix)   :", " ".join(str(round(r['f_N2'], 6)) for r in res_mix))